問題：撰寫一Python程式，為使用者輸入年份（可能是西元或民國）找尋其相對應的十二生肖。十二生肖是以十二年為一週期，每一年分別以一種動物來表示猴（monkey）、雞（rooster）、狗(dog)、豬(pig)、鼠(rat)、牛(ox)、虎(tiger)、兔(rabbit)、龍(dragon)、蛇（snake）、馬(horse)，以及羊(sheep) 。

請依序回答下面問題：

1. 0.1的二進位表示法為何? 使用markdown語法，列出計算過程和理由，並描述看到的現象。

將十進位小數轉換為二進位採用乘 2 取整數法：計算步驟計算式整數部分剩餘小數部分第 1 步$0.1 \times 2 = 0.2$0$0.2$第 2 步$0.2 \times 2 = 0.4$0$0.4$第 3 步$0.4 \times 2 = 0.8$0$0.8$第 4 步$0.8 \times 2 = 1.6$1$0.6$第 5 步$0.6 \times 2 = 1.2$1$0.2$第 6 步$0.2 \times 2 = 0.4$0$0.4$（回到第 2 步）
現象與理由：依序取整數部分，結果為：$$0.0001100110011..._2$$看到的現象：自小數第 2 位開始，0011 會無窮循環。理由：二進位小數能被精確表示的充分必要條件是該數可化簡為分母為 $2^n$ 的最簡分數。因為 $0.1 = \frac{1}{10} = \frac{1}{2 \times 5}$，分母含有質因數 5，故無法在二進位中有限表示，形成無窮循環小數。

2. 如果是無限多位，Python如何解決？提示：IEEE 754、四捨六入五成雙

電腦硬體記憶體有限，Python 遵循 IEEE 754 雙精度浮點數（Double Precision, 64-bit） 規範來處理：

位元分配：

符號位（Sign）：1 bit

指數位（Exponent）：11 bit

尾數位/有效位（Fraction/Mantissa）：52 bit（隱含最高位 1，實際具 53 bit 精度）

捨入機制（四捨六入五成雙 / Round half to even）：

當二進位無窮循環截斷到第 53 位時，依照「捨入到最接近的值；若剛好落在正中間，則捨入到最後一位為偶數（二進位的 0）」的原則。

這種機制可避免傳統「四捨五入」在大量計算時累積單向的正向統計偏差。

3. Python如何知道浮點數的精度？請問執行以下程式碼:
```python
import sys
print(sys.float_info)
```
並說明輸出結果的意思。

import sys
print(sys.float_info)

sys.float_info(max=1.7976931348623157e+308, max_exp=1024, max_10_exp=308, min=2.2250738585072014e-308, min_exp=-1021, min_10_exp=-307, dig=15, mant_dig=53, epsilon=2.220446049250313e-16, radix=2, rounds=1)

mant_dig=53：二進位尾數有效位數為 53 位元。dig=15：保證轉換為十進位時不會失真的有效位數（約 15 至 17 位十進位數字）。epsilon=2.220446049250313e-16：機器精度（Machine Epsilon），即大於 1 的最小浮點數與 1 之間的差值（$2^{-52}$），代表浮點數能分辨的最小相對差異。max / min：雙精度浮點數能表示的最大與最小正規化正數值。radix=2：數值基數為二進位。rounds=1：捨入模式為 IEEE 754 的捨入至最接近值（Round to nearest）。

４. 在數學上0.1+0.2=0.3，可是下面的Python程式的答案好像不是0.3。怎麼解決？
```python
print(0.1+0.2)
```

import math
print(math.isclose(0.1 + 0.2, 0.3))  # True
from decimal import Decimal
# 需傳入字串避免預先轉為二進位浮點數
result = Decimal('0.1') + Decimal('0.2')
print(result)          # 0.3
print(result == Decimal('0.3'))  # True
from fractions import Fraction
print(Fraction('0.1') + Fraction('0.2') == Fraction('0.3'))  # True

5. 你的方法，對於一般兩個浮點數加法，在科學或金融上是好方法嗎？請思考。

金融領域（Financial Domain）
評估：使用標準硬體浮點數（float）是極具風險且不被允許的；使用 Decimal 是標準正確做法。

理由：

金融法規對帳務、利率、結算要求「美分/角級別的絕對精確」，不允許出現 0.0000000000000004 這類捨入漂移。

Decimal 以十進位為基底儲存，完全貼合貨幣的計算法規與明確的進位模式（如金融 Banker's rounding）。

科學與工程領域（Scientific Computing）
評估：使用 Decimal 是低效且不合適的；使用 IEEE 754 float（如 float64 / numpy.float64）配合容差比較是最佳解法。

理由：

硬體加速：現代 CPU/GPU 具備專屬的浮點運算單元（FPU/SIMD），硬體 float 計算速度比軟體模擬的 Decimal 快上百倍甚至千倍。

容錯性：科學計算的輸入數據本身就帶有物理測量不確定度（如感測器誤差），計算著重於有效位數的控制與數值穩定性，透過演算法避免災難性抵消（Catastrophic Cancellation），並使用 math.isclose() 或 np.isclose() 判定相等即可。